# Task 4: Context-Aware Chatbot Using LangChain + RAG
**DevelopersHub Corporation – AI/ML Engineering Internship**

## Problem Statement & Objective
Build a conversational chatbot that:
1. Maintains **conversation memory** across turns
2. **Retrieves relevant context** from a custom document corpus using vector search (RAG)
3. Is deployed via an interactive **Streamlit** interface

## 1. Install Dependencies

In [ ]:
!pip install langchain langchain-community langchain-openai langchain-huggingface \
             chromadb sentence-transformers streamlit openai faiss-cpu tiktoken \
             wikipedia pypdf -q

## 2. Imports & Configuration

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WikipediaLoader, TextLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.memory import ConversationBufferWindowMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_community.llms import HuggingFaceHub
from langchain_openai import ChatOpenAI
from langchain.schema import Document
import textwrap

# --- CONFIGURATION ---
# Option A: Use OpenAI (recommended for quality)
# os.environ['OPENAI_API_KEY'] = 'your-key-here'

# Option B: Use HuggingFace (free)
# os.environ['HUGGINGFACEHUB_API_TOKEN'] = 'your-token-here'

EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'  # Free, fast, accurate
PERSIST_DIR     = './chroma_db'

print('All imports successful!')

## 3. Dataset Loading & Document Corpus Construction

In [ ]:
# Load documents from Wikipedia (custom knowledge base)
# Modify topics to fit your use case
TOPICS = [
    'Artificial intelligence',
    'Machine learning',
    'Natural language processing',
    'Large language model',
    'Neural network'
]

print(f'Loading {len(TOPICS)} Wikipedia articles...')
all_docs = []

for topic in TOPICS:
    try:
        loader = WikipediaLoader(query=topic, load_max_docs=1)
        docs   = loader.load()
        all_docs.extend(docs)
        print(f'  ✓ Loaded: {topic} ({len(docs[0].page_content)} chars)')
    except Exception as e:
        print(f'  ✗ Failed: {topic} — {e}')

print(f'\nTotal documents loaded: {len(all_docs)}')
print(f'Total characters: {sum(len(d.page_content) for d in all_docs):,}')

In [ ]:
# --- OPTIONAL: Add your own text files to the corpus ---
# custom_loader = TextLoader('./my_document.txt')
# all_docs.extend(custom_loader.load())

# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # characters per chunk
    chunk_overlap=150,     # overlap to preserve context across chunks
    separators=['\n\n', '\n', '. ', ' ', '']
)

chunks = text_splitter.split_documents(all_docs)
print(f'Total chunks created: {len(chunks)}')
print(f'Average chunk size:   {sum(len(c.page_content) for c in chunks)//len(chunks)} chars')
print(f'\nSample chunk preview:\n{chunks[5].page_content[:300]}...')

## 4. Vector Store Construction (Embeddings + ChromaDB)

In [ ]:
# Initialize the embedding model
print('Loading embedding model...')
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cpu'},   # Use 'cuda' if GPU available
    encode_kwargs={'normalize_embeddings': True}
)

# Test embedding
test_embed = embeddings.embed_query('What is machine learning?')
print(f'Embedding model loaded. Vector dimension: {len(test_embed)}')

In [ ]:
# Create & persist vector store
print('Building vector store (this may take a moment)...')

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIR
)

print(f'Vector store built and persisted at: {PERSIST_DIR}')
print(f'Total vectors stored: {vectorstore._collection.count()}')

In [ ]:
# Test retrieval
retriever = vectorstore.as_retriever(
    search_type='mmr',       # Maximal Marginal Relevance — reduces redundancy
    search_kwargs={'k': 4, 'fetch_k': 10}
)

test_query = 'How does a transformer model work?'
results    = retriever.invoke(test_query)

print(f'Query: "{test_query}"')
print(f'Retrieved {len(results)} chunks:\n')
for i, doc in enumerate(results):
    print(f'[{i+1}] Source: {doc.metadata.get("source", "Unknown")}')
    print(f'     Preview: {doc.page_content[:150]}...')
    print()

## 5. LLM Setup & Conversational Chain

In [ ]:
# --- Choose your LLM ---

# OPTION A: OpenAI GPT (best quality, requires API key)
# llm = ChatOpenAI(
#     model='gpt-3.5-turbo',
#     temperature=0.3,
#     openai_api_key=os.environ['OPENAI_API_KEY']
# )

# OPTION B: HuggingFace Mistral (free, requires HF token)
from langchain_community.llms import HuggingFaceHub
llm = HuggingFaceHub(
    repo_id='mistralai/Mistral-7B-Instruct-v0.2',
    huggingfacehub_api_token=os.environ.get('HUGGINGFACEHUB_API_TOKEN', ''),
    model_kwargs={'temperature': 0.3, 'max_new_tokens': 512}
)

print('LLM initialized!')

In [ ]:
# Conversation memory — keeps last 5 exchanges in context
memory = ConversationBufferWindowMemory(
    memory_key='chat_history',
    return_messages=True,
    output_key='answer',
    k=5
)

# Build the full RAG conversational chain
qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    memory=memory,
    return_source_documents=True,
    verbose=False
)

print('ConversationalRetrievalChain ready!')

## 6. Test the Chatbot in Notebook

In [ ]:
def chat(question):
    """Send a question to the RAG chatbot and display the answer with sources."""
    print(f'\n🧑 USER: {question}')
    response = qa_chain.invoke({'question': question})
    answer   = response['answer']
    sources  = response.get('source_documents', [])

    print(f'\n🤖 BOT: {textwrap.fill(answer, width=90)}')

    if sources:
        unique_sources = list({d.metadata.get('source','Unknown') for d in sources})
        print(f'\n📚 Sources: {" | ".join(unique_sources)}')
    print('-' * 80)
    return answer

# Multi-turn conversation test
chat('What is machine learning?')
chat('How is it different from deep learning?')  # Tests memory — 'it' refers to ML
chat('What are transformers and why are they important?')

## 7. Streamlit App
Save the cell below as `app.py` and run with `streamlit run app.py`

In [ ]:
streamlit_app = '''
import streamlit as st
import os
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.memory import ConversationBufferWindowMemory
from langchain.chains import ConversationalRetrievalChain
from langchain_community.llms import HuggingFaceHub

st.set_page_config(page_title="RAG Chatbot", page_icon="🤖", layout="wide")
st.title("🤖 Context-Aware RAG Chatbot")
st.caption("Powered by LangChain + ChromaDB + Mistral-7B")

@st.cache_resource
def load_chain():
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = Chroma(persist_directory="./chroma_db", embedding_function=embeddings)
    retriever   = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 4})
    llm = HuggingFaceHub(
        repo_id="mistralai/Mistral-7B-Instruct-v0.2",
        huggingfacehub_api_token=os.environ.get("HUGGINGFACEHUB_API_TOKEN", ""),
        model_kwargs={"temperature": 0.3, "max_new_tokens": 512}
    )
    memory = ConversationBufferWindowMemory(
        memory_key="chat_history", return_messages=True, output_key="answer", k=5
    )
    return ConversationalRetrievalChain.from_llm(
        llm=llm, retriever=retriever, memory=memory,
        return_source_documents=True, verbose=False
    )

chain = load_chain()

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

if prompt := st.chat_input("Ask me anything about AI..."):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            response = chain.invoke({"question": prompt})
            answer   = response["answer"]
            sources  = response.get("source_documents", [])
            st.write(answer)
            if sources:
                with st.expander("📚 Sources"):
                    for doc in sources:
                        st.caption(f"• {doc.metadata.get('source', 'Unknown')}: {doc.page_content[:200]}...")

    st.session_state.messages.append({"role": "assistant", "content": answer})
'''

with open('app.py', 'w') as f:
    f.write(streamlit_app.strip())
print('Streamlit app saved as app.py')
print('Run: streamlit run app.py')

## 8. Final Summary & Insights

### Architecture Overview
```
User Query
    ↓
Embed query (MiniLM-L6-v2)
    ↓
MMR Retrieval from ChromaDB  ← Wikipedia Corpus (chunked)
    ↓
ConversationBufferWindowMemory (last 5 turns)
    ↓
Mistral-7B / GPT → Answer + Sources
```

### Key Design Decisions
| Decision | Rationale |
|----------|-----------|
| Chunk size: 1000 chars | Balances context richness vs retrieval noise |
| MMR retrieval | Reduces redundant chunks; improves answer diversity |
| Window memory (k=5) | Maintains context without blowing up token budget |
| all-MiniLM-L6-v2 | Fast, free, 384-dim, excellent semantic search quality |

### Key Insights
- RAG dramatically reduces LLM hallucinations by grounding answers in retrieved facts.
- Conversation memory allows pronoun resolution ('it', 'that model') across turns.
- ChromaDB persistence means the vector store is built once and reloaded cheaply.
- Chunk overlap (150 chars) prevents context loss at chunk boundaries.